# Stage 7b: Embeddings


In [ ]:
!pip install -q sentence-transformers torch

In [ ]:
import os
import json
import glob
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'BAAI/bge-large-en-v1.5'
WORKING_DIR = '/kaggle/working'

if not glob.glob(os.path.join(WORKING_DIR, '*.jsonl')):
    print("Didn't find files in /kaggle/working. Searching /kaggle/input...")
    inputs = glob.glob('/kaggle/input/**/*.jsonl', recursive=True)
    if inputs:
        WORKING_DIR = os.path.dirname(inputs[0])

all_files = glob.glob(os.path.join(WORKING_DIR, '*.jsonl'))
official_files = [f for f in all_files if 'api_knowledge' in f or 'guides_knowledge' in f]

def load_files(files_list):
    metadata, texts = [], []
    for fpath in files_list:
        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                data = json.loads(line)
                texts.append(data.get('knowledge', ''))
                metadata.append(data)
    return texts, metadata

print(f'Loading data from {WORKING_DIR}...')
texts_official, meta_official = load_files(official_files)
texts_community, meta_community = load_files(all_files)

print(f'Official Chunks: {len(texts_official)} | Community Chunks: {len(texts_community)}')

if len(texts_community) == 0:
    print("⚠️ NO DATA FOUND! Please upload the .jsonl files to Kaggle.")
else:
    print('Loading model...')
    model = SentenceTransformer(MODEL_NAME)
    
    print('Generating Official Embeddings...')
    emb_official = model.encode(texts_official, convert_to_tensor=True, show_progress_bar=True)
    torch.save(emb_official, '/kaggle/working/embeddings_official.pt')
    with open('/kaggle/working/metadata_official.json', 'w', encoding='utf-8') as f:
        json.dump(meta_official, f)
        
    print('Generating Community Embeddings...')
    emb_community = model.encode(texts_community, convert_to_tensor=True, show_progress_bar=True)
    torch.save(emb_community, '/kaggle/working/embeddings_community.pt')
    with open('/kaggle/working/metadata_community.json', 'w', encoding='utf-8') as f:
        json.dump(meta_community, f)
    
    print('✅ Done! Download embeddings_official.pt, metadata_official.json, embeddings_community.pt, and metadata_community.json')